# 🎲 Mosaico com dados D6 — prova de conceito (Colab)

PoC mínimo do mosaico com dados D6:

1. Cinza → redimensionar para grade (1 pixel = 1 dado)
2. **Binarização** (Otsu) na grade
3. **Quantização** em faces **1** (claro) … **6** (escuro)
4. **Rotação** opcional nas faces **2**, **3** e **6**
5. Pré-visualização e matriz de montagem

> Abra no [Google Colab](https://colab.research.google.com/) e execute as células em ordem.

In [ ]:
# Dependências (Colab já traz NumPy; OpenCV e Matplotlib instalam rápido)
!pip install -q opencv-python-headless matplotlib

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

# Se você clonou o repositório, o módulo está em colab/; senão copie mosaic_poc.py para o Colab
COLAB_DIR = Path(".")
if (COLAB_DIR / "mosaic_poc.py").exists():
    sys.path.insert(0, str(COLAB_DIR.resolve()))
elif (COLAB_DIR / "colab" / "mosaic_poc.py").exists():
    sys.path.insert(0, str((COLAB_DIR / "colab").resolve()))

from mosaic_poc import (
    MosaicConfig,
    build_mosaic,
    load_image_from_bytes,
    render_mosaic_bgr,
    rotations_summary,
    suggest_grid_size,
)

%matplotlib inline

In [ ]:
# Upload da imagem de teste
uploaded = files.upload()
if not uploaded:
    raise SystemExit("Envie um arquivo de imagem (PNG/JPG).")
filename = next(iter(uploaded))
image_bytes = uploaded[filename]
source_bgr = load_image_from_bytes(image_bytes)
h, w = source_bgr.shape[:2]
print(f"Imagem: {filename} — {w}×{h} px")

In [ ]:
# --- Parâmetros do PoC (ajuste aqui) ---
DICE_PX = max(8, min(w, h) // 40)  # pixels da fonte por dado (maior = mosaico mais grosseiro)
dice_cols, dice_rows = suggest_grid_size(w, h, DICE_PX)

OPTIMIZE_ROTATION = True  # False desliga rotação das faces 2, 3 e 6
PREVIEW_CELL = 32  # tamanho de cada dado na pré-visualização (px)

print(f"Grade: {dice_cols} colunas × {dice_rows} linhas ≈ {dice_cols * dice_rows:,} dados")

In [ ]:
config = MosaicConfig(
    dice_cols=dice_cols,
    dice_rows=dice_rows,
    optimize_rotation=OPTIMIZE_ROTATION,
)

result = build_mosaic(source_bgr, config)
mosaic_bgr = render_mosaic_bgr(
    result.faces, result.rotations, cell_size=PREVIEW_CELL
)

print(rotations_summary(result.rotations, result.faces))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(cv2.cvtColor(source_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(result.grid_gray, cmap="gray")
axes[1].set_title(f"Grade cinza {dice_cols}×{dice_rows}")
axes[1].axis("off")

axes[2].imshow(result.grid_binary, cmap="gray")
axes[2].set_title("Binarização (Otsu)")
axes[2].axis("off")

axes[3].imshow(cv2.cvtColor(mosaic_bgr, cv2.COLOR_BGR2RGB))
axes[3].set_title("Mosaico quantizado" + (" + rotação" if OPTIMIZE_ROTATION else ""))
axes[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Comparação: com vs sem rotação (útil se houver faces 2, 3 ou 6)
if OPTIMIZE_ROTATION:
    cfg_no_rot = MosaicConfig(
        dice_cols=dice_cols,
        dice_rows=dice_rows,
        optimize_rotation=False,
    )
    without = build_mosaic(source_bgr, cfg_no_rot)
    img_with = render_mosaic_bgr(result.faces, result.rotations, PREVIEW_CELL)
    img_without = render_mosaic_bgr(without.faces, without.rotations, PREVIEW_CELL)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(img_without, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Sem otimizar rotação")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(img_with, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Com rotação nas faces 2, 3 e 6")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Matriz de montagem: face + rotação (só exibe ° onde face ∈ {2,3,6})
rows, cols = result.faces.shape
print(f"Montagem — {cols}×{rows} ({cols * rows} dados)")
print("Formato: face ou face@graus (ex.: 3@90 = face 3 girada 90°)")
print("=" * 50)
for r in range(min(rows, 24)):  # limita linhas no log do Colab
    parts = []
    for c in range(cols):
        f = int(result.faces[r, c])
        rot = int(result.rotations[r, c])
        if f in (2, 3, 6) and rot:
            parts.append(f"{f}@{rot * 90}")
        else:
            parts.append(str(f))
    print(f"R{r:03d}: " + " ".join(parts))
if rows > 24:
    print(f"... (+{rows - 24} linhas; use result.faces / result.rotations no código)")

## Pipeline

| Etapa | Função |
|-------|--------|
| Cinza + resize | Um valor por célula da grade |
| `binarize_otsu` | Limiar automático → preto/branco (visualização) |
| `quantize_to_faces` | 6 faixas de cinza → faces 1..6 (montagem) |
| `optimize_rotations` | Faces 2/3/6: melhor rotação por MSE com template |

A quantização usa a grade **em cinza**; a binarização mostra o mesmo conteúdo em 2 níveis (Otsu por célula na grade reduzida).